# Fault Tolerance

Kafi Streams ensures fault tolerance with checkpointing.

We explain how it works in [Checkpointing](#checkpointing).

Finally, we show checkpointing in action by a practical [example](#example).

Notice that fault tolerance is only supported by `Streams`, not the `TopologyNode` class as only the former includes support for persistence (through Kafi).


## Overview

[Preparation](#prep)

* [Checkpointing](#checkpointing)
  * [Enabling checkpointing](#enabling)
  * [Checkpointing in detail](#detail)
* [Example](#example)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [ ]:
import sys
sys.path.insert(1, "../..")

from kafi.streams.streams import Streams

from generators import OrderGenerator
order_generator = OrderGenerator()

order_source_str = "orders"
sink_str = "sink"


---
<a id="checkpointing"></a>
## Checkpointing

How does checkpointing work in Kafi Streams?

This global state can be stored in any *storage* supported by Kafi, i.e., currently, Kafka itself, or, via Kafi's *emulated Kafka*, to disk, S3 or Azure Blob Storage.

[Enabling checkpointing](#enabling) explains how to enable checkpointing, and [Checkpointing in detail](#detail) describes how checkpointing is implemented in detail and embedded into the consume + process + produce loop of the `Streams` class.


<a id="enabling"></a>
### Enabling checkpointing

Let's revisit the signature of the `start_streams` method of the `Streams` class to see how checkpointing can be enabled:
```python
@staticmethod
def start_streams(built_tn, checkpoint_storage=None, checkpoint_topic_str=None, checkpoint_interval_float=default_checkpoint_interval_float, **kwargs):
    """Run streams() in a background thread; returns a function to stop it.

    Args:
        built_tn: built tn to run
        checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
        checkpoint_topic_str: topic name used to store checkpoints
        checkpoint_interval_float: seconds between checkpoints
        **kwargs: passed through to streams()
    Returns:
        stop_fun: None -> None function to stop the Streams processing thread"""
```

You can enable checkpointing by setting `checkpoint_storage` and `checkpoint_topic_str` to the Kafi stroage and topic to be used for the checkpointing.

The `checkpoint_interval_float` is a floating point number specifying the checkpoint interval (in seconds).


<a id="detail"></a>
### Checkpointing in detail

`Streams` implements the typical consume + process + produce loop from stream processing. Let's look at it in more detail in the case if checkpointing is enabled:

* before the loop: read the last checkpoint if there is any and set the global state and the offsets of the consumer group for the source topics accordingly,
* in the loop:
  1. consume new data from the source topics,
  2. process the new data and return the new resulting data,
  3. produce the new resulting data to the sink topics.
  4. if there is new resulting data and the checkpoint interval is exceeded:  
     4.1 save the checkpoint = the current global state + the offsets of the last processed messages from the source topics,  
     4.2 commit these offsets to Kafka.


---
<a id="example"></a>
## Example

This section shows an example of checkpointing in Kafi Streams.


<a id="topology"></a>
### Topology

The topology has one source (orders) and aggregates these orders per `customer_id`:
* `orders` the number of orders of the customer,
* `order_ids` the `order_id`s of the customer,
* `total_price` the sum of the `price`s of the orders of the customer:


In [ ]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.kafka.cluster.cluster import Cluster
from kafi.streams.streams import Streams

import logging
logging.basicConfig(level=logging.DEBUG)

c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

source_str = "orders"
sink_str = "orders_aggregated"

sink_tn = (
    Streams.source(c, source_str)

    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "order_ids": sorted(agg_r["order_ids"] + [r["order_id"]]),
                                    "total_price": agg_r["total_price"] + r["price"]},
                  {"orders": 0, "order_ids": [], "total_price": 0},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "order_ids": agg_r["order_ids"],
                                     "total_price": agg_r["total_price"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)

tn = Streams.build(sink_tn)



<a id="step_1"></a>
### Step 1

In step 1, we:
* start the Streams thread using the `start_streams()` method (with checkpointing enabled),
* generate `1000` orders, produce them to the source topic and let the Streams thread process them:

In [ ]:
from kafi.helpers import get_millis

orders_int = 1000

tn.reset()

checkpoint_str = "checkpoint"
g = f"group_{get_millis()}"

c.recreate(source_str)
c.recreate(sink_str)
c.recreate(checkpoint_str)

#

stop_fun = Streams.start_streams(tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_str, checkpoint_interval_floatrval=0.01, group=g)

#

pr = c.producer(source_str)
gen = OrderGenerator()

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



As you see in the output, a first checkpoint has been saved to the checkpoint topic.

<a id="step_2"></a>
### Step 2

In step 2, we stop the Streams thread:

In [ ]:
stop_fun()
Streams.threads()

<a id="step_3"></a>
### Step 3

In step 3, we:
* reset the state of the built topology node (`tn`),
* (re-)start the *Streams* thread using the `start_streams()` method (with checkpointing enabled),
* generate another `1000` orders, produce them to the source topic and let Streams thread process them:

In [ ]:
tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams(tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_str, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



In the output log, you can see that the checkpoint from the previous processing ([step 1](#step_1)) is correctly loaded and thus the state recovered.

<a id="step_4"></a>
### Step 4

In step 4, we stop the *Streams* thread once again:

In [ ]:
stop_fun()
Streams.threads()

<a id="step_5"></a>
### Step 5

In step 5, we again:
* reset the state of the built topology node (`tn`),
* (re-)start the *Streams* thread using the `start_streams()` method (with checkpointing enabled),
* generate another `1000` orders, produce them to the source topic and let Streams thread process them:

In [ ]:
tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams(tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_str, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()


<a id="step_6"></a>
### Step 6

In step 6, we stop the Streams thread a last time:


In [ ]:
stop_fun()
Streams.threads()

<a id="step_7"></a>
### Step 7

In the last step 7, we:
* read both the source and the sink topic,
* calculate the aggregations outside of Kafi Streams,
* and compare them to the aggregations read from the sink topic:

In [ ]:
import math

source_key_int_value_dict_dict = {}
source_m_list = c.cat(source_str)
for m in source_m_list:
    customer_id_int = m["value"]["customer_id"]
    order_id_str = m["value"]["order_id"]
    price_int = m["value"]["price"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_order_id_str_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("order_ids", [])
    agg_total_price_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("total_price", 0)
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "order_ids": sorted(agg_order_id_str_list + [order_id_str]),
                                                       "total_price": agg_total_price_int + price_int}

#

sink_key_int_value_dict_dict = {}
sink_m_list = c.cat(sink_str)
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["value"]["customer_id"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

for key_int, value_dict in source_key_int_value_dict_dict.items():
    keys_match_bool = value_dict["customer_id"] == sink_key_int_value_dict_dict[key_int]["customer_id"]
    orders_match_bool = value_dict["orders"] == sink_key_int_value_dict_dict[key_int]["orders"]
    order_ids_match_bool = value_dict["order_ids"] == sink_key_int_value_dict_dict[key_int]["order_ids"]
    price_match_bool = math.isclose(
        value_dict["total_price"], sink_key_int_value_dict_dict[key_int]["total_price"], rel_tol=1e-9)
    #
    if not (keys_match_bool and orders_match_bool and order_ids_match_bool and price_match_bool):
        print("First mismatch:")
        print("Source:", source_dict)
        print("Sink:  ", sink_dict)
        raise Exception("Test failed")
#

print("Test successful.")


Because in this step, we read the entire source topic which has been subsequently filled with new data after we had intentionally stopped the Streams thread, and the entire sink topic, and because all the source data is randomly generated, this steps show that the checkpointing in Kafi Streams works :)
